# 01 · Data, Leakage Audit and Lesion-Grouped Splits

This notebook prepares everything the later notebooks depend on:

1. loads HAM10000 and summarises the class and lesion distribution;
2. **audits data leakage** in a conventional image-level split, where several photographs of the same lesion can end up in both training and test data;
3. creates the **lesion-grouped, stratified 5-fold cross-validation** used for every result in this project, with an inner lesion-grouped validation split;
4. verifies the folds automatically and exports them to `splits/lesion_grouped_folds.csv`;
5. caches all images at 224 × 224 so that training and evaluation decode each image only once.

**Inputs:** HAM10000 images and `HAM10000_metadata.csv` · **Outputs:** `outputs/splits/cv_splits.json`, `outputs/cache/images_ham10000_224.*`, `splits/lesion_grouped_folds.csv`

Existing splits and caches are reused, never overwritten, so the folds used for the published models stay fixed.

## Setup

Paths can be edited in the next cell or set as environment variables (`HAM_PROJECT_OUT`, `HAM10000_DIR`, `ISIC2019_DIR`).

In [ ]:
import os, json, math, time, random, platform, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:                                   # progress bars are optional
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

In [ ]:
# ---------------------------------------------------------------- paths (edit here or set the environment variables)
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

def env_path(name, default):
    return Path(os.environ.get(name, str(default))).expanduser()

PROJECT_OUT  = env_path('HAM_PROJECT_OUT', r'D:/HAM_PROJECT/outputs_v2')   # caches, checkpoints, features, tables (not in Git)
HAM_DIR      = env_path('HAM10000_DIR', r'D:/HAM_PROJECT/dataset')      # HAM10000 images, metadata and lesion masks
ISIC2019_DIR = env_path('ISIC2019_DIR', r'D:/Skin Dataset/ISIC2019')    # ISIC 2019 training images, ground truth, metadata
RESULTS_DIR  = env_path('RESULTS_DIR', REPO / 'results')             # tables and figures shown in the README
SPLITS_DIR   = env_path('SPLITS_DIR', REPO / 'splits')               # published fold assignment

# ---------------------------------------------------------------- study settings (keep unchanged to reproduce the results)
CLASSES   = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
MALIGNANT = ['akiec', 'bcc', 'mel']                                  # (pre)malignant classes
MODELS    = ['efficientnet_b4', 'convnext_base', 'densenet121']
LABEL_OF  = {'efficientnet_b4': 'EfficientNet-B4', 'convnext_base': 'ConvNeXt-Base', 'densenet121': 'DenseNet-121'}
IMG_SIZE, SEED, N_FOLDS = 224, 42, 5
DROPOUT, MC_SAMPLES = 0.4, 30                                       # head dropout; MC Dropout passes per model
UNC_QUANTILE, GMM_N, N_NVREF = 0.75, 3, 1000                        # uncertainty flag; HBDS levels; nevus reference size
VARIANT = 'focal_ls'                                                # main models (focal loss + label smoothing)
EXT_MC_SEED0 = 555                                                  # MC seed offset for external data

C = len(CLASSES)
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
MAL_IDX = [CLASS2IDX[c] for c in MALIGNANT]
EPS = 1e-12
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DIRS = {k: PROJECT_OUT / k for k in ['splits', 'checkpoints', 'cache', 'tables', 'figures']}
for _d in list(DIRS.values()) + [RESULTS_DIR, SPLITS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def mean_sd(values, digits=4):
    s = pd.Series(values, dtype=float).dropna()
    return f'{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}' if len(s) > 1 else (f'{s.mean():.{digits}f}' if len(s) else 'n/a')

print(f'Python {platform.python_version()} | PyTorch {torch.__version__} | device: {DEVICE}')
print(f'Project outputs: {PROJECT_OUT}')

In [ ]:
import cv2
from PIL import Image

def find_file(root, name):
    """Locate a dataset file by name anywhere below root."""
    direct = Path(root) / name
    if direct.is_file():
        return direct
    hits = sorted(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f"'{name}' not found under {root}. See the Dataset section of README.md.")
    return hits[0]

def image_index(root, exclude=('segmentation', 'mask')):
    """Map image stem -> path for all images below root (lesion masks excluded)."""
    index = {}
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and not any(e in p.as_posix().lower() for e in exclude):
            index.setdefault(p.stem, p)
    return index

def read_rgb(path):
    with Image.open(path) as im:
        return np.array(im.convert('RGB'))

def read_rgb_224(path):
    """Pipeline image loading: PIL -> RGB -> cv2 bilinear resize to 224 x 224."""
    return cv2.resize(read_rgb(path), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

def build_image_cache(paths, npy_path, json_path, desc='caching images'):
    """Decode every image once into a uint8 array (N, 224, 224, 3). Reused if it already exists."""
    paths = [str(p) for p in paths]
    if Path(npy_path).exists() and Path(json_path).exists():
        cached = json.load(open(json_path))
        if [Path(p).stem for p in cached] == [Path(p).stem for p in paths]:
            return np.load(npy_path, mmap_mode='r')
        raise RuntimeError(f'{json_path} exists but lists different images - delete it to rebuild the cache.')
    Path(npy_path).parent.mkdir(parents=True, exist_ok=True)
    arr = np.lib.format.open_memmap(npy_path, mode='w+', dtype=np.uint8, shape=(len(paths), IMG_SIZE, IMG_SIZE, 3))
    for i, p in enumerate(tqdm(paths, desc=desc)):
        arr[i] = read_rgb_224(p)
    arr.flush(); del arr
    json.dump(paths, open(json_path, 'w'))
    return np.load(npy_path, mmap_mode='r')

def load_ham():
    """HAM10000 metadata in the order of the image cache, labels and lesion groups."""
    meta = pd.read_csv(find_file(HAM_DIR, 'HAM10000_metadata.csv')).drop_duplicates('image_id').set_index('image_id')
    order_file = DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.json'
    if not order_file.exists():
        raise FileNotFoundError(f'{order_file} not found - run notebook 01 first.')
    ids = [Path(p).stem for p in json.load(open(order_file))]
    ham = meta.loc[ids].reset_index()
    return ham, ham['dx'].map(CLASS2IDX).astype(int).values, ham['lesion_id'].astype(str).values

def load_splits():
    f = DIRS['splits'] / 'cv_splits.json'
    if not f.exists():
        raise FileNotFoundError(f'{f} not found - run notebook 01 first.')
    return {int(k): {kk: np.asarray(vv, dtype=int) for kk, vv in v.items()} for k, v in json.load(open(f)).items()}

## 1. Dataset overview

In [ ]:
meta = pd.read_csv(find_file(HAM_DIR, 'HAM10000_metadata.csv')).drop_duplicates('image_id').reset_index(drop=True)
summary = (meta.groupby('dx').agg(images=('image_id', 'size'), lesions=('lesion_id', 'nunique')).reindex(CLASSES))
summary.insert(0, 'type', ['(pre)malignant' if c in MALIGNANT else 'benign' for c in summary.index])
summary['images per lesion'] = (summary['images'] / summary['lesions']).round(2)
summary['share of images (%)'] = (100 * summary['images'] / summary['images'].sum()).round(1)
summary.loc['total'] = ['', summary['images'].sum(), meta['lesion_id'].nunique(),
                        round(len(meta) / meta['lesion_id'].nunique(), 2), 100.0]
display(summary)
summary.to_csv(DIRS['tables'] / 'dataset_summary.csv')

multi = meta.groupby('lesion_id').size()
print(f'{len(meta)} images of {meta.lesion_id.nunique()} lesions; {int((multi > 1).sum())} lesions '
      f'({100 * (multi > 1).mean():.1f}%) have more than one image, covering '
      f'{100 * meta.lesion_id.map(multi).gt(1).mean():.1f}% of all images.')

fig, ax = plt.subplots(figsize=(7, 3.6))
counts = summary.loc[CLASSES, 'images'].astype(int)
ax.bar(CLASSES, counts, color=['#c0392b' if c in MALIGNANT else '#2e86c1' for c in CLASSES])
for i, v in enumerate(counts):
    ax.text(i, v + 60, f'{v:,}', ha='center', fontsize=8)
ax.set_ylabel('images'); ax.set_title('HAM10000 class distribution (red: (pre)malignant, blue: benign)', fontsize=10)
ax.grid(alpha=.3, axis='y'); plt.tight_layout(); plt.savefig(DIRS['figures'] / 'class_distribution.png', dpi=150); plt.show()

## 2. Leakage audit of a conventional image-level split

A common protocol splits *images* 70/15/15 at random (stratified by class). Because many lesions were photographed several times,
this places views of the same lesion in both training and test data. The cell below reproduces such a split (seed 42) and counts the
test images whose lesion also appears in the training or validation partition.

In [ ]:
from sklearn.model_selection import train_test_split
idx = np.arange(len(meta))
trval, test = train_test_split(idx, test_size=0.15, stratify=meta['dx'], random_state=SEED)
train, val = train_test_split(trval, test_size=0.15 / 0.85, stratify=meta['dx'].values[trval], random_state=SEED)
seen = set(meta['lesion_id'].values[np.r_[train, val]])
leaked = meta['lesion_id'].values[test]
leak_mask = np.array([l in seen for l in leaked])
leak = pd.DataFrame({'partition': ['train', 'validation', 'test'], 'images': [len(train), len(val), len(test)]})
display(leak)
print(f'Test images sharing a lesion with training/validation images: {leak_mask.sum()} of {len(test)} '
      f'({100 * leak_mask.mean():.1f}%)')
pd.DataFrame({'test images': [len(test)], 'leaked test images': [int(leak_mask.sum())],
              'leaked share (%)': [round(100 * leak_mask.mean(), 1)]}).to_csv(DIRS['tables'] / 'leakage_audit.csv', index=False)

## 3. Image cache

Every image is decoded once (PIL → RGB → bilinear resize to 224 × 224) and stored as a uint8 array. The cache order defines the
row indices used in `cv_splits.json`.

In [ ]:
npy, js = DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.npy', DIRS['cache'] / f'images_ham10000_{IMG_SIZE}.json'
if js.exists():
    cache_paths = json.load(open(js))
    print(f'Reusing existing image cache ({len(cache_paths)} images).')
elif (DIRS['splits'] / 'cv_splits.json').exists():
    raise RuntimeError('cv_splits.json exists but the image cache it indexes is missing; copy images_ham10000_224.json/.npy '
                       'from the original output folder (rebuilding would change the row order).')
else:
    index = image_index(HAM_DIR)
    order = meta.sort_values('image_id')['image_id'].tolist()
    missing = [i for i in order if i not in index]
    if missing:
        raise FileNotFoundError(f'{len(missing)} HAM10000 images not found under {HAM_DIR}, e.g. {missing[:3]}')
    cache_paths = [index[i] for i in order]
HAM_IMGS = build_image_cache(cache_paths, npy, js, desc='caching HAM10000')
ham, LABELS, GROUPS = load_ham()
print('cache:', HAM_IMGS.shape, HAM_IMGS.dtype)

## 4. Lesion-grouped, stratified 5-fold cross-validation

`StratifiedGroupKFold` with `lesion_id` as the group keeps every lesion inside a single partition. In each fold, the held-out fifth
of lesions forms the test set, and one of six inner lesion-grouped groups of the remaining data forms the validation set (used for
early stopping, calibration and every fitted threshold).

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
split_file = DIRS['splits'] / 'cv_splits.json'
if split_file.exists():
    SPLITS = load_splits()
    print(f'Reusing existing folds from {split_file}')
else:
    SPLITS = {}
    outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k, (trval_idx, test_idx) in enumerate(outer.split(np.zeros(len(LABELS)), LABELS, GROUPS)):
        inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
        tr_rel, va_rel = next(inner.split(np.zeros(len(trval_idx)), LABELS[trval_idx], GROUPS[trval_idx]))
        SPLITS[k] = {'train': np.sort(trval_idx[tr_rel]), 'val': np.sort(trval_idx[va_rel]), 'test': np.sort(test_idx)}
    json.dump({str(k): {p: v.tolist() for p, v in s.items() if p in ('train', 'val', 'test')} for k, s in SPLITS.items()},
              open(split_file, 'w'))
    print(f'Created {split_file}')
FOLDS = sorted(SPLITS)

### Automatic checks

In [ ]:
tested = np.zeros(len(LABELS), int)
rows = []
for k in FOLDS:
    sp = SPLITS[k]
    parts = {p: set(GROUPS[sp[p]]) for p in ('train', 'val', 'test')}
    overlap = len(parts['train'] & parts['val']) + len(parts['train'] & parts['test']) + len(parts['val'] & parts['test'])
    assert overlap == 0, f'fold {k}: {overlap} lesions appear in more than one partition'
    assert len(set(sp['train']) | set(sp['val']) | set(sp['test'])) == len(sp['train']) + len(sp['val']) + len(sp['test'])
    tested[sp['test']] += 1
    rows.append({'fold': k, 'train': len(sp['train']), 'validation': len(sp['val']), 'test': len(sp['test']),
                 **{f'test {c}': int((LABELS[sp['test']] == i).sum()) for i, c in enumerate(CLASSES)}})
assert (tested == 1).all(), 'every image must be tested exactly once'
fold_table = pd.DataFrame(rows).set_index('fold'); display(fold_table)
fold_table.to_csv(DIRS['tables'] / 'fold_composition.csv')
print('PASS: no lesion is shared between partitions of any fold, and every image is tested exactly once.')

## 5. Export the published fold assignment

In [ ]:
export = pd.DataFrame({'image_id': ham['image_id'], 'lesion_id': GROUPS, 'dx': ham['dx']})
export['test_fold'] = -1
for k in FOLDS:
    role = np.full(len(export), 'unused', dtype=object)
    for p in ('train', 'val', 'test'):
        role[SPLITS[k][p]] = p
    export[f'fold{k}_role'] = role
    export.loc[SPLITS[k]['test'], 'test_fold'] = k
export.to_csv(SPLITS_DIR / 'lesion_grouped_folds.csv', index=False)
print(f"Saved {SPLITS_DIR / 'lesion_grouped_folds.csv'} ({len(export)} rows)")
display(export.head())

## Summary

* The image-level split leaks lesions into the test set (see the audit above); all results in this project therefore use the
  lesion-grouped folds created here.
* The fold assignment is published in `splits/lesion_grouped_folds.csv` (image and lesion identifiers only, no images).